In [1]:
import os
import glob
import pandas as pd
import rasterio
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# grid + fire + soil

In [3]:
all_points = pd.read_csv("/content/drive/MyDrive/collab/f2/soil_merged_cleaned_without_duplicate.csv")

In [4]:
climat = pd.read_csv("/content/drive/MyDrive/collab/f2/climat.csv")

In [5]:
print("Nombre de points de sol :", len(all_points))
print("Nombre de points de climat :", len(climat))

Nombre de points de sol : 89198
Nombre de points de climat : 89198


In [11]:
climat.isna().sum()

,0
longitude,0
latitude,0
prec_hiver,6
prec_printemps,6
prec_ete,6
prec_automne,6
tmin_hiver,6
tmin_printemps,6
tmin_ete,6
tmin_automne,6


In [12]:
climat[climat.isna().any(axis=1)]

,longitude,latitude,prec_hiver,prec_printemps,prec_ete,prec_automne,tmin_hiver,tmin_printemps,tmin_ete,tmin_automne,tmax_hiver,tmax_printemps,tmax_ete,tmax_automne
1,-8.952,36.994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26659,-9.502,38.782,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41954,-9.167,39.506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43537,-9.087,39.606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43673,-9.087,39.615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51009,-8.917,40.003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
from sklearn.neighbors import BallTree

col = ['prec_hiver', 'prec_printemps', 'prec_ete',
       'prec_automne', 'tmin_hiver', 'tmin_printemps', 'tmin_ete',
       'tmin_automne', 'tmax_hiver', 'tmax_printemps', 'tmax_ete',
       'tmax_automne']
valid = ~climat[col].isna().any(axis=1)
nan = ~valid
# coordonnees en radians longitude	latitude
valid_coords = np.radians(climat.loc[valid, ["longitude", "latitude"]].values)
nan_coords = np.radians(climat.loc[nan, ["longitude", "latitude"]].values)

# arbre
tree = BallTree(valid_coords, metric="haversine")

# indice du plus proche voisin
dist, idx = tree.query(nan_coords, k=1)

# remplacer par la valeur la plus proche
climat.loc[nan,col] = climat.loc[valid,col].values[idx[:, 0]]

In [14]:
climat[climat.isna().any(axis=1)]

,longitude,latitude,prec_hiver,prec_printemps,prec_ete,prec_automne,tmin_hiver,tmin_printemps,tmin_ete,tmin_automne,tmax_hiver,tmax_printemps,tmax_ete,tmax_automne


In [15]:

# --- paramètres ---
seuil_km = 1.0
seuil_rad = seuil_km / 6371.0  # rayon Terre en km

# --- coords en radians (lat, lon) ---
all_rad = np.radians(all_points[["latitude", "longitude"]].values)
climat_rad = np.radians(climat[["latitude", "longitude"]].values)

# --- arbre sur climat ---
tree = BallTree(climat_rad, metric="haversine")

# --- nearest neighbor pour chaque point ---
dist, idx = tree.query(all_rad, k=1)

# distance en km
dist_km = dist[:, 0] * 6371.0

# --- merge par index du plus proche ---
climat_cols = climat.columns.difference(["longitude", "latitude"])
merged = all_points.copy()
merged[climat_cols] = climat.iloc[idx[:, 0]][climat_cols].values

# --- filtrer par seuil ---
merged["dist_climat_km"] = dist_km
merged["match_climat"] = (dist[:, 0] <= seuil_rad).astype(int)

# --- stats ---
print("Total all_points:", len(all_points))
print("Match <= seuil:", merged["match_climat"].sum())
print("No match:", (merged["match_climat"] == 0).sum())
print("Distance min/max (km):", merged["dist_climat_km"].min(), merged["dist_climat_km"].max())

# optionnel: mettre NaN si pas de match
merged.loc[merged["match_climat"] == 0, climat_cols] = np.nan

merged.drop(columns=["dist_climat_km", "match_climat"], inplace=True)

Total all_points: 89198
Match <= seuil: 89198
No match: 0
Distance min/max (km): 0.0003512594419692697 0.07052210857647025


In [16]:
merged.to_csv("/content/drive/MyDrive/collab/f2/soil_climat_fire_grid_merged.csv", index=False)

In [17]:
Elevation = pd.read_csv("/content/drive/MyDrive/collab/f2/elevation.csv")

In [19]:
Elevation.shape

(89198, 3)

In [18]:
Elevation.isna().sum()

,0
longitude,0
latitude,0
Elevation,0


In [20]:
# --- paramètres ---
seuil_km = 1.0
seuil_rad = seuil_km / 6371.0

# --- coords en radians (lat, lon) ---
all_rad = np.radians(merged[["latitude", "longitude"]].values)
elev_rad = np.radians(Elevation[["latitude", "longitude"]].values)

# --- arbre sur elevation ---
tree = BallTree(elev_rad, metric="haversine")

# --- nearest neighbor pour chaque point ---
dist, idx = tree.query(all_rad, k=1)

# distance en km
dist_km = dist[:, 0] * 6371.0

# --- merge par index du plus proche ---
elev_cols = Elevation.columns.difference(["longitude", "latitude"])
merged_elev = merged.copy()
merged_elev[elev_cols] = Elevation.iloc[idx[:, 0]][elev_cols].values

# --- filtrer par seuil ---
merged_elev["dist_elev_km"] = dist_km
merged_elev["match_elev"] = (dist[:, 0] <= seuil_rad).astype(int)

# --- stats ---
print("Total all_points:", len(merged))
print("Match <= seuil:", merged_elev["match_elev"].sum())
print("No match:", (merged_elev["match_elev"] == 0).sum())
print("Distance min/max (km):", merged_elev["dist_elev_km"].min(), merged_elev["dist_elev_km"].max())
 
# optionnel: mettre NaN si pas de match
merged_elev.loc[merged_elev["match_elev"] == 0, elev_cols] = np.nan

merged_elev.drop(columns=["dist_elev_km", "match_elev"], inplace=True)

Total all_points: 89198
Match <= seuil: 89198
No match: 0
Distance min/max (km): 0.0 0.0


In [21]:
merged_elev.to_csv("/content/drive/MyDrive/collab/f2/soil_climat_fire_grid_Elev_merged.csv", index=False)

In [3]:
merged_elev = pd.read_csv("/content/drive/MyDrive/collab/f2/soil_climat_fire_grid_Elev_merged.csv")

In [4]:
merged_elev.shape

(89198, 38)

In [5]:
merged_elev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89198 entries, 0 to 89197
Data columns (total 38 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   latitude        89198 non-null  float64
 1   longitude       89198 non-null  float64
 2   landcover       89198 non-null  int64  
 3   SMU_ID          89198 non-null  int64  
 4   HWSD2_SMU_ID    89198 non-null  float64
 5   COARSE          89198 non-null  float64
 6   SAND            89198 non-null  float64
 7   SILT            89198 non-null  float64
 8   CLAY            89198 non-null  float64
 9   TEXTURE_USDA    89198 non-null  float64
 10  TEXTURE_SOTER   89198 non-null  object 
 11  BULK            89198 non-null  float64
 12  ORG_CARBON      89198 non-null  float64
 13  PH_WATER        89198 non-null  float64
 14  CN_RATIO        89198 non-null  float64
 15  CEC_SOIL        89198 non-null  float64
 16  CEC_CLAY        89198 non-null  float64
 17  CEC_EFF         89198 non-null 

In [6]:


df_corr = merged_elev.select_dtypes(include=[np.number])


corr_matrix = df_corr.corr(method='pearson').abs()


upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))


seuil = 0.9


high_corr = [(column, row) for row in upper.index for column in upper.columns if upper.loc[row, column] > seuil]


print(f"Couples avec une corrélation > {seuil} :")
for col, row in high_corr:
    val = upper.loc[row, col]
    print(f"{row} <-> {col} : {val:.2f}")

Couples avec une corrélation > 0.9 :
latitude <-> prec_ete : 0.94
prec_automne <-> prec_ete : 0.96
prec_automne <-> prec_hiver : 0.92
prec_automne <-> prec_printemps : 0.94
prec_ete <-> prec_hiver : 0.90
prec_ete <-> prec_printemps : 0.91
prec_hiver <-> prec_printemps : 0.91
tmax_automne <-> tmax_hiver : 0.94
tmax_automne <-> tmax_printemps : 0.99
tmax_hiver <-> tmax_printemps : 0.95
tmax_hiver <-> tmin_automne : 0.95
tmax_hiver <-> tmin_hiver : 0.92
tmax_hiver <-> tmin_printemps : 0.93
tmin_automne <-> tmin_hiver : 0.97
tmin_automne <-> tmin_printemps : 0.97
tmin_hiver <-> tmin_printemps : 0.98


In [7]:
merged_elev["dif_perc_hiver_printemps"] = merged_elev["prec_hiver"] - merged_elev["prec_printemps"]

In [8]:
merged_elev["dif_perc_automne_printemps"] = merged_elev["prec_automne"] - merged_elev["prec_printemps"]

In [9]:
merged_elev["dif_perc_ete_automne"] = merged_elev["prec_ete"] - merged_elev["prec_automne"]

In [10]:
merged_elev.drop(columns=["prec_hiver", "prec_printemps", "prec_automne"], inplace=True)

In [11]:
merged_elev["dif_max_automne_printemps"] = merged_elev["tmax_automne"] - merged_elev["tmax_printemps"]

In [12]:
merged_elev["dif_max_hiver_printemps"] = merged_elev["tmax_hiver"] - merged_elev["tmax_printemps"]

In [13]:
merged_elev.drop(columns=["tmax_automne", "tmax_printemps"], inplace=True)

In [14]:
merged_elev["dif_min_hiver_printemps"] = merged_elev["tmin_hiver"] - merged_elev["tmin_printemps"]

In [15]:
merged_elev["dif_min_automne_hiver"] = merged_elev["tmin_automne"] - merged_elev["tmin_hiver"]

In [16]:
merged_elev["dif_max_hiver_min_automne"] = merged_elev["tmax_hiver"] - merged_elev["tmin_automne"]

In [17]:
merged_elev.drop(columns=["tmin_automne", "tmin_printemps", "tmin_hiver"], inplace=True)

In [18]:


df_corr = merged_elev.select_dtypes(include=[np.number])


corr_matrix = df_corr.corr(method='pearson').abs()


upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))


seuil = 0.9


high_corr = [(column, row) for row in upper.index for column in upper.columns if upper.loc[row, column] > seuil]


print(f"Couples avec une corrélation > {seuil} :")
for col, row in high_corr:
    val = upper.loc[row, col]
    print(f"{row} <-> {col} : {val:.2f}")

Couples avec une corrélation > 0.9 :
latitude <-> prec_ete : 0.94
prec_ete <-> dif_perc_ete_automne : 0.91


In [19]:
merged_elev.isna().sum()

,0
latitude,0
longitude,0
landcover,0
SMU_ID,0
HWSD2_SMU_ID,0
COARSE,0
SAND,0
SILT,0
CLAY,0
TEXTURE_USDA,0


In [20]:
merged_elev.drop(columns=["SMU_ID","HWSD2_SMU_ID"], inplace=True)   

In [21]:
merged_elev.to_csv("/content/drive/MyDrive/collab/f2/all_merged_csv_without_correlation.csv", index=False)